In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Form_file_fasta)=
# file:fasta

*FASTA sequence file format*

- **Technical Form Name:** `file:fasta`
- **Form Type:** `file`

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

- Form adapter module: {mod}`molsysmt.form.file_fasta`
:::

## Overview

`file:fasta` is a supported representation form in MolSysMT. FASTA sequence file format

The UniProt FASTA adapter owns an automatically generated output until writing
and closing succeed. Failure retires that generated file. Successful output
belongs to you. An explicit output path remains yours and may contain partial
content after a failed write; cleanup errors remain visible. High-level
{func}`molsysmt.basic.convert` requires an explicit output filename for file
conversions.


## Supported attributes

The `file:fasta` form container supports a total of **15 attributes**:

| Category | Subcategory / Level | Supported Attributes |
|---|---|---|
| **Topological** | Group Level | `group_index`, `group_name`, `group_type` |
| **Topological** | Chain Level | `chain_index`, `chain_name`, `chain_id`, `chain_type` |
| **Topological** | Entity Level | `entity_index`, `entity_name`, `entity_id`, `entity_type` |
| **Topological** | Element Counts | `n_groups`, `n_chains`, `n_entities`, `n_amino_acids` |


Chain and entity indices are zero-based source record positions. For a two-record file, `msm.get(molsys, element="chain", selection=[1, 0, 1], chain_index=True)` returns `[1, 0, 1]`; the corresponding entity query uses `element="entity", entity_index=True`. Record IDs remain string labels. Empty files or empty selections return empty lists. These queries use the existing optional Biopython sequence reader and construct no atom topology.

## Implemented operations

The form adapter module {mod}`molsysmt.form.file_fasta` implements the following functions and modules:

| Function / Module | Description |
|---|---|
| `parse_records(fasta_text, source_label=None)` | Inspects text with no normalization and retains every raw record and source character span. |
| `read_records(item)` | Reads strict UTF-8 without newline translation into the same source-preserving result; no Biopython is required. |
| {func}`~molsysmt.form.file_fasta.extract.extract` | Extracts a subset of atoms, groups, or structures into a new `file:fasta` instance. |
| {func}`~molsysmt.form.file_fasta.copy.copy` | Creates an independent deep copy of the `file:fasta` object in memory. |
| {func}`~molsysmt.form.file_fasta.add.add` | Adds elements from another system into an existing `file:fasta` instance. |
| {func}`~molsysmt.form.file_fasta.merge.merge` | Merges multiple systems into a single combined `file:fasta` object. |
| {mod}`~molsysmt.form.file_fasta.get_topological_attributes` | Module containing getters for topological attributes. |
| {mod}`~molsysmt.form.file_fasta.get_structural_attributes` | Module containing getters for structural attributes. |
| {mod}`~molsysmt.form.file_fasta.set` | Module containing setters for updating coordinates, box dimensions, or attributes in-place. |
| {mod}`~molsysmt.form.file_fasta.iterators` | Module providing iterators for streaming trajectory frames and structures. |
| {func}`~molsysmt.form.file_fasta.has_attribute.has_attribute` | Checks whether a specific attribute name is supported by `file:fasta`. |
| {func}`~molsysmt.form.file_fasta.is_form.is_form` | Returns `True` if the provided object is an instance of `file:fasta`. |

:::{note}
:class: dropdown

Each form adapter in MolSysMT implements only the specific functions that are meaningful for its data model.
:::

### Preserving submitted records

Use `msm.form.file_fasta.parse_records(text)` or
`msm.form.file_fasta.read_records(path)` before scientific admission. The
JSON-compatible `molsysmt.fasta_source@1` dictionary contains an opaque
`source` label, verbatim blank `preamble` and ordered `records`. Each record
has a zero-based `record_index`, complete `header`, joined unnormalized
`sequence`, `raw_record` and half-open `source_span` character offsets into
the decoded input. These are source occurrences, not group indices or IDs.

Joining the preamble and raw records reconstructs the complete decoded input.
Only CR/LF/CRLF line terminators are omitted from the sequence view: case,
spaces, tabs, leading M, stops and modification syntax remain unchanged.
Repeated and empty records/headers remain separate. Nonblank text before a
header rejects explicitly; comments after a header remain sequence content.
There is no chemical/alphabet admission, implicit normalization or topology
construction. The resident result retains raw and joined views in memory.

The ordinary `msm.convert(path, to_form="biopython.SeqRecord")` route retains
its existing normalized behavior and optional Biopython dependency. Choose
that conversion only after deciding whether its normalization is appropriate.
The complete return contract and executed examples are documented below.

```{eval-rst}
.. autofunction:: molsysmt.form.file_fasta.parse_records

.. autofunction:: molsysmt.form.file_fasta.read_records
```


## Supported conversions

The following table lists all direct conversion functions implemented in {mod}`molsysmt.form.file_fasta`:

| Target Form | Form Type | API Documentation |
|---|---|---|
| `biopython.Seq` | `class` | {func}`~molsysmt.form.file_fasta.to_biopython_Seq.to_biopython_Seq` |
| `biopython.SeqRecord` | `class` | {func}`~molsysmt.form.file_fasta.to_biopython_SeqRecord.to_biopython_SeqRecord` |
| `file:fasta` | `file` | {func}`~molsysmt.form.file_fasta.to_file_fasta.to_file_fasta` |
| `molsysmt.Topology` | `class` | {func}`~molsysmt.form.file_fasta.to_molsysmt_Topology.to_molsysmt_Topology` |
| `string:amino_acids_1` | `string` | {func}`~molsysmt.form.file_fasta.to_string_amino_acids_1.to_string_amino_acids_1` |
| `molsysmt.MolSys` | `class` | {func}`~molsysmt.form.file_fasta.to_molsysmt_MolSys.to_molsysmt_MolSys` |

### Native group inventories

Topology and MolSys conversion now retain one declared molecule/entity/chain
slot per record, including duplicate labels/sequences and empty records. They
populate groups and group-to-molecule membership, with string IDs and no stored
atoms, bonds or coordinate structures. This replaces the former implicit
one-CA-per-position representation. The existing amino-acid interpretation
is retained; overlapping nucleotide/peptide alphabets are not automatically
disambiguated. Separate entity slots do not establish
chemical inequality or deduplicate molecular identities. The existing optional
Biopython reader supplies normalized sequence text and complete descriptions;
`read_records` remains the independent source-preserving route.

A FASTA record is a molecular occurrence, not a coordinate structure. Atomic
and structure selections are unsupported on this sequence-only conversion.
Unknown one-letter symbols, gaps and stops need explicit interpretation before
molecular projection. The original source document remains unchanged.

Native chain IDs are generated as `A`, `B`, ..., `Z`, `AA`, `AB`, and onward;
record IDs are retained as names and complete headers as entity names. Direct
FASTA chain-ID queries use the same alphabetic convention. Source record IDs
remain available through the raw reader and direct entity-ID queries.

The compact ordered chain/group relation survives copy, TopologyDict and H5MSM
0.5 round trips, including empty and repeated record occurrences. Native text
selectors and group/chain/molecule/entity projections operate without atoms.
Use `msm.extract(..., element="group", selection=[2, 0])` to reorder groups,
or `element="chain"` to retain whole chains, including empty chains. A single
one-letter output string concatenates sequences and cannot encode those
boundaries. No atomic-realization claim follows from these conversions.

```{eval-rst}
.. autofunction:: molsysmt.form.file_fasta.to_molsysmt_Topology.to_molsysmt_Topology

.. autofunction:: molsysmt.form.file_fasta.to_molsysmt_MolSys.to_molsysmt_MolSys
```


:::{seealso}
:class: dropdown

**Foundations & Object Architecture:**
- {ref}`user-foundations-molecular-system-definition`: The fundamental definition of a molecular system, forms, and attributes in MolSysMT.
- {ref}`user-foundations-native-world-classes-molsysmt-molsys`: In-depth architectural guide on the native `molsysmt.MolSys` container.
- {ref}`user-foundations-native-world-classes-molsysmt-topology`: Architectural specification of the native `Topology` object.
- {ref}`user-foundations-native-world-classes-molsysmt-structures`: Architectural specification of the native `Structures` object.

**Form Tools & Conversions:**
- {ref}`Convert <Tutorial_Convert>`: Converting between any molecular system representations with {func}`molsysmt.basic.convert`.
- {ref}`Get attributes <Tutorial_Form_Get_attributes>`: Querying all supported attributes for any form with {func}`molsysmt.form.get_attributes`.
- {ref}`Has attribute <Tutorial_Form_Has_attribute>`: Testing attribute availability for a form with {func}`molsysmt.form.has_attribute`.
:::
